<a href="https://colab.research.google.com/github/daslk1128-ops/10th-toy-team4/blob/main/gbis_data_quickstart_saebom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GBIS 데이터 불러오기 (팀원용)

이 노트북은 개발 환경 설정 없이 GBIS 데이터를 Google Colab에서 DataFrame으로 불러옵니다.

처음 한 번만 다음 작업을 해주세요.

1. Colab 왼쪽의 **열쇠(Secrets)** 아이콘을 누릅니다.
2. 이름이 `GBIS_API_KEY`인 새 보안 비밀을 만들고 전달받은 개인 키를 입력합니다.
3. 이 노트북에서 해당 보안 비밀에 대한 **Notebook access**를 켭니다.
4. 상단 메뉴에서 **런타임 → 모두 실행**을 누릅니다.

최초 실행은 전체 이력을 내려받으므로 시간이 걸릴 수 있습니다. 이후에는 Google Drive의 캐시를 복원하고 새 데이터만 받습니다.

In [ ]:
# 아래 값은 특별한 경우가 아니면 그대로 사용하세요.
API_BASE_URL = "https://161.33.212.6"  # @param {type:"string"}
DRIVE_CACHE_PATH = "/content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3"  # @param {type:"string"}
ROUTE_IDS = ""  # @param {type:"string"}
HISTORY_FROM = ""  # @param {type:"string"}

import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

try:
    API_KEY = userdata.get("GBIS_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "왼쪽 열쇠 아이콘에서 GBIS_API_KEY를 만들고 Notebook access를 켜주세요."
    ) from exc
if not API_KEY:
    raise RuntimeError("Colab Secrets의 GBIS_API_KEY가 비어 있습니다.")

REPO_DIR = Path("/content/gbis_team_repo")
REPO_URL = "https://github.com/khuda-data/10th-toy-team4.git"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO_DIR / "requirements-client.txt"),
    ],
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

drive.mount("/content/drive")
DRIVE_CACHE = Path(DRIVE_CACHE_PATH)
LOCAL_CACHE = Path("/content/gbis_api_cache.sqlite3")
DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
CACHE_RESTORED = DRIVE_CACHE.is_file()
if CACHE_RESTORED:
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
    print("✅ Google Drive에서 기존 캐시를 복원했습니다.")
else:
    print("ℹ️ 첫 실행입니다. 서버의 전체 이력을 내려받습니다.")

Mounted at /content/drive
✅ Google Drive에서 기존 캐시를 복원했습니다.


In [ ]:
from IPython.display import display
from gbis_client import GBISApiCache

requested_route_ids = tuple(
    value.strip() for value in ROUTE_IDS.split(",") if value.strip()
)
cache = GBISApiCache(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    cache_path=LOCAL_CACHE,
)
history_counts = {}
try:
    print("1/4 노선 목록을 최신화합니다.")
    cache.refresh_routes()
    routes_df = cache.routes_df()
    available_route_ids = set(routes_df["route_id"].astype(str))
    if requested_route_ids:
        missing_route_ids = set(requested_route_ids) - available_route_ids
        if missing_route_ids:
            raise ValueError(f"서버에 없는 route_id입니다: {sorted(missing_route_ids)}")
        route_ids = requested_route_ids
    else:
        route_ids = tuple(str(value) for value in routes_df["route_id"].tolist())

    print("2/4 정류장 정보를 최신화합니다.")
    station_count_by_route = dict(zip(routes_df["route_id"].astype(str), routes_df["station_count"]))
    station_counts = {
        route_id: cache.refresh_stations(route_id)
        for route_id in route_ids
        if int(station_count_by_route.get(route_id, 0)) > 0
    }

    print("3/4 최신 차량 위치를 갱신합니다.")
    cache.refresh_latest()

    print("4/4 차량 위치 이력을 동기화합니다.")
    for index, route_id in enumerate(route_ids, 1):
        mode = "증분" if CACHE_RESTORED else "최초 전체"
        print(f"  [{index}/{len(route_ids)}] {route_id}: {mode} 동기화 중...")
        history_counts[route_id] = cache.refresh_full_history(route_id)

    routes_df = cache.routes_df()
    stations_df = cache.stations_df()
    latest_df = cache.latest_locations_df()
    history_df = cache.history_df(from_at=HISTORY_FROM or None)
    cache_status_df = cache.cache_status_df()
    if requested_route_ids:
        stations_df = stations_df[stations_df["route_id"].isin(route_ids)].reset_index(drop=True)
        latest_df = latest_df[latest_df["route_id"].isin(route_ids)].reset_index(drop=True)
        history_df = history_df[history_df["route_id"].isin(route_ids)].reset_index(drop=True)
finally:
    cache.close()
    if LOCAL_CACHE.is_file():
        shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)
        print(f"✅ 캐시를 Google Drive에 저장했습니다: {DRIVE_CACHE}")

print("\n동기화 완료")
print(f"- routes_df: {len(routes_df):,}행")
print(f"- stations_df: {len(stations_df):,}행")
print(f"- latest_df: {len(latest_df):,}행")
print(f"- history_df: {len(history_df):,}행")
display(history_df.head())

1/4 노선 목록을 최신화합니다.
2/4 정류장 정보를 최신화합니다.
3/4 최신 차량 위치를 갱신합니다.
4/4 차량 위치 이력을 동기화합니다.
  [1/17] 200000104: 증분 동기화 중...
  [2/17] 204000057: 증분 동기화 중...
  [3/17] 218000010: 증분 동기화 중...
  [4/17] 219000013: 증분 동기화 중...
  [5/17] 219000016: 증분 동기화 중...
  [6/17] 222000074: 증분 동기화 중...
  [7/17] 222000075: 증분 동기화 중...
  [8/17] 222000209: 증분 동기화 중...
  [9/17] 228000174: 증분 동기화 중...
  [10/17] 229000311: 증분 동기화 중...
  [11/17] 232000090: 증분 동기화 중...
  [12/17] 234000309: 증분 동기화 중...
  [13/17] 234000878: 증분 동기화 중...
  [14/17] 234001243: 증분 동기화 중...
  [15/17] 234001245: 증분 동기화 중...
  [16/17] 234001695: 증분 동기화 중...
  [17/17] 234001736: 증분 동기화 중...
✅ 캐시를 Google Drive에 저장했습니다: /content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3

동기화 완료
- routes_df: 17행
- stations_df: 1,220행
- latest_df: 210행
- history_df: 499,272행


,route_id,vehicle_id,observed_at,query_time,plate_no,route_type_code,station_id,station_seq,station_name,remaining_seats,crowded,low_plate,state_code,tagless_code,cached_at_utc
0,219000013,218000030,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아3485,11,219000561,52,대화역(중),39,1,0,0,0,2026-08-11T17:05:29+00:00
1,219000013,218000160,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1117,11,100000034,27,광화문역6번출구.광화문빌딩,41,1,0,2,1,2026-08-11T17:05:29+00:00
2,219000013,218000161,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1118,11,277103099,26,경복궁역(경유),43,1,0,2,1,2026-08-11T17:05:29+00:00
3,219000013,218000162,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1120,11,112000012,23,연세대앞(중),33,1,0,2,1,2026-08-11T17:05:29+00:00
4,219000013,218000165,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1134,11,277103096,21,증산교교차로(경유),34,1,0,0,1,2026-08-11T17:05:29+00:00


## 사용할 수 있는 DataFrame

- `routes_df`: 노선 목록과 수집 범위
- `stations_df`: 노선별 정류장 순서와 위치
- `latest_df`: 현재 운행 차량별 최신 위치와 잔여좌석
- `history_df`: 전체 또는 지정 시각 이후의 차량 위치 이력
- `cache_status_df`: 데이터별 마지막 갱신 완료 시각

예를 들어 잔여좌석이 0인 기록은 `history_df[history_df["remaining_seats"] == 0]`으로 확인할 수 있습니다.

In [ ]:
routes_df.head()

,route_id,station_count,observation_count,first_collected_at,last_collected_at,cached_at_utc
0,200000104,86,51157,2026-08-09T19:36:01+09:00,2026-08-14T21:34:03+09:00,2026-08-14T12:34:32+00:00
1,204000057,85,1396,2026-08-03T13:22:01+09:00,2026-08-03T21:58:02+09:00,2026-08-14T12:34:32+00:00
2,218000010,96,57597,2026-08-09T19:36:01+09:00,2026-08-14T21:34:03+09:00,2026-08-14T12:34:32+00:00
3,219000013,55,225780,2026-08-03T13:22:00+09:00,2026-08-14T21:34:02+09:00,2026-08-14T12:34:32+00:00
4,219000016,77,62034,2026-08-09T19:36:01+09:00,2026-08-14T21:34:02+09:00,2026-08-14T12:34:32+00:00


In [ ]:
print("===== 데이터 크기 =====")
print("routes_df :", routes_df.shape)
print("stations_df :", stations_df.shape)
print("latest_df :", latest_df.shape)
print("history_df :", history_df.shape)

print("\n===== history_df 컬럼 =====")
print(history_df.columns.tolist())

print("\n===== 결측치 =====")
print(history_df.isnull().sum())

print("\n===== 잔여석 통계 =====")
print(history_df["remaining_seats"].describe())

print("\n===== 만차 비율 =====")
print((history_df["remaining_seats"] == 0).mean())

display(history_df.head())

===== 데이터 크기 =====
routes_df : (17, 6)
stations_df : (1220, 11)
latest_df : (210, 15)
history_df : (499272, 15)

===== history_df 컬럼 =====
['route_id', 'vehicle_id', 'observed_at', 'query_time', 'plate_no', 'route_type_code', 'station_id', 'station_seq', 'station_name', 'remaining_seats', 'crowded', 'low_plate', 'state_code', 'tagless_code', 'cached_at_utc']

===== 결측치 =====
route_id             0
vehicle_id           0
observed_at          0
query_time           0
plate_no             0
route_type_code      0
station_id           0
station_seq          0
station_name       170
remaining_seats      0
crowded              0
low_plate            0
state_code           0
tagless_code         0
cached_at_utc        0
dtype: int64

===== 잔여석 통계 =====
count    499272.000000
mean         34.419839
std          12.001733
min          -1.000000
25%          30.000000
50%          38.000000
75%          42.000000
max          70.000000
Name: remaining_seats, dtype: float64

===== 만차 비율 =====
0.0

,route_id,vehicle_id,observed_at,query_time,plate_no,route_type_code,station_id,station_seq,station_name,remaining_seats,crowded,low_plate,state_code,tagless_code,cached_at_utc
0,219000013,218000030,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아3485,11,219000561,52,대화역(중),39,1,0,0,0,2026-08-11T17:05:29+00:00
1,219000013,218000160,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1117,11,100000034,27,광화문역6번출구.광화문빌딩,41,1,0,2,1,2026-08-11T17:05:29+00:00
2,219000013,218000161,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1118,11,277103099,26,경복궁역(경유),43,1,0,2,1,2026-08-11T17:05:29+00:00
3,219000013,218000162,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1120,11,112000012,23,연세대앞(중),33,1,0,2,1,2026-08-11T17:05:29+00:00
4,219000013,218000165,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1134,11,277103096,21,증산교교차로(경유),34,1,0,0,1,2026-08-11T17:05:29+00:00


In [ ]:
print("===== 잔여좌석별 개수 =====")
print(history_df["remaining_seats"].value_counts().sort_index())

print("\n===== 0석 비율 =====")
print((history_df["remaining_seats"] == 0).mean())

print("\n===== -1 비율 =====")
print((history_df["remaining_seats"] == -1).mean())

print("\n===== -1 개수 =====")
print((history_df["remaining_seats"] == -1).sum())

===== 잔여좌석별 개수 =====
remaining_seats
-1     1126
 0     9351
 1     1820
 2     1566
 3     1802
       ... 
 66    1050
 67    1236
 68    1349
 69    1680
 70    1633
Name: count, Length: 72, dtype: int64

===== 0석 비율 =====
0.018729269816853338

===== -1 비율 =====
0.002255283693057091

===== -1 개수 =====
1126


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. 기본 전처리
# --------------------------------------------------

df = history_df.copy()

# 시간 변환
df["observed_at"] = pd.to_datetime(df["observed_at"])

# 잔여좌석 -1은 정보 없음으로 보고 제외
df = df[df["remaining_seats"] >= 0].copy()

# 시간순 정렬
df = df.sort_values(
    ["route_id", "vehicle_id", "observed_at"]
).reset_index(drop=True)

# 시간대
df["hour"] = df["observed_at"].dt.hour


# --------------------------------------------------
# 2. 미래 정류장의 잔여좌석을 target으로 만들기
# --------------------------------------------------

# 같은 버스의 미래 기록을 찾기 위해
# 차량 + 날짜 기준으로 그룹화
df["date"] = df["observed_at"].dt.date

group_cols = ["route_id", "vehicle_id", "date"]

# 최대 5개 정류장 뒤의 잔여좌석을 target으로 사용
for step in range(1, 6):
    df[f"target_{step}"] = (
        df.groupby(group_cols)["remaining_seats"]
        .shift(-step)
    )

# 우선 3개 정류장 뒤를 예측하는 모델로 설정
df["target"] = df["target_3"]


# --------------------------------------------------
# 3. target이 존재하는 데이터만 사용
# --------------------------------------------------

model_df = df.dropna(subset=["target"]).copy()

# target도 실제 잔여좌석이어야 함
model_df = model_df[model_df["target"] >= 0].copy()


# --------------------------------------------------
# 4. 남은 정류장 수
# --------------------------------------------------

model_df["remaining_stations"] = 3


# --------------------------------------------------
# 5. 사용할 Feature
# --------------------------------------------------

features = [
    "remaining_seats",
    "station_seq",
    "remaining_stations",
    "crowded",
    "hour"
]

X = model_df[features]
y = model_df["target"]

print("학습 데이터:", model_df.shape)
print("Feature:", features)
print("Target: 3개 정류장 뒤 잔여좌석")

학습 데이터: (494488, 24)
Feature: ['remaining_seats', 'station_seq', 'remaining_stations', 'crowded', 'hour']
Target: 3개 정류장 뒤 잔여좌석


In [ ]:
# 날짜 기준으로 시간순 분리

dates = sorted(model_df["date"].unique())

print("데이터 날짜:", dates)

# 마지막 날짜는 Validation으로 사용
valid_date = dates[-1]

train_df = model_df[model_df["date"] < valid_date].copy()
valid_df = model_df[model_df["date"] == valid_date].copy()

X_train = train_df[features]
y_train = train_df["target"]

X_valid = valid_df[features]
y_valid = valid_df["target"]

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

데이터 날짜: [datetime.date(2026, 8, 3), datetime.date(2026, 8, 4), datetime.date(2026, 8, 5), datetime.date(2026, 8, 6), datetime.date(2026, 8, 7), datetime.date(2026, 8, 8), datetime.date(2026, 8, 9), datetime.date(2026, 8, 10), datetime.date(2026, 8, 11), datetime.date(2026, 8, 12), datetime.date(2026, 8, 13), datetime.date(2026, 8, 14)]
Train: (425033, 5)
Validation: (69455, 5)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_depth=15
)

model.fit(X_train, y_train)

pred = model.predict(X_valid)

print("모델 학습 완료!")

모델 학습 완료!


In [ ]:
from sklearn.metrics import mean_absolute_error

# 실제 잔여석이 10석 이하인 데이터만 선택
low_mask = y_valid <= 10

low_mae = mean_absolute_error(
    y_valid[low_mask],
    pred[low_mask]
)

print("===== 저잔여석 MAE =====")
print(f"MAE: {low_mae:.3f}석")
print(f"대상 데이터 수: {low_mask.sum()}개")

===== 저잔여석 MAE =====
MAE: 2.109석
대상 데이터 수: 4433개


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 실제 만차
y_true_full = (y_valid == 0).astype(int)

# 예측 만차
y_pred_full = (pred < 0.5).astype(int)

accuracy = accuracy_score(y_true_full, y_pred_full)
precision = precision_score(
    y_true_full,
    y_pred_full,
    zero_division=0
)
recall = recall_score(
    y_true_full,
    y_pred_full,
    zero_division=0
)
f1 = f1_score(
    y_true_full,
    y_pred_full,
    zero_division=0
)

print("===== 만차 여부 평가 =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")

===== 만차 여부 평가 =====
Accuracy : 0.9929
Precision: 0.9557
Recall   : 0.7180
F1       : 0.8200


In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print("===== Feature Importance =====")
display(importance)

===== Feature Importance =====


,0
remaining_seats,0.970919
station_seq,0.017802
hour,0.010424
crowded,0.000856
remaining_stations,0.000000
